In [1]:
import pandas as pd

df = pd.read_csv("../data/sales_data.csv")

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicates:", df.duplicated().sum())

Shape: (76000, 16)

Columns:
['Date', 'Store ID', 'Product ID', 'Category', 'Region', 'Inventory Level', 'Units Sold', 'Units Ordered', 'Price', 'Discount', 'Weather Condition', 'Promotion', 'Competitor Pricing', 'Seasonality', 'Epidemic', 'Demand']

Missing values:
Date                  0
Store ID              0
Product ID            0
Category              0
Region                0
Inventory Level       0
Units Sold            0
Units Ordered         0
Price                 0
Discount              0
Weather Condition     0
Promotion             0
Competitor Pricing    0
Seasonality           0
Epidemic              0
Demand                0
dtype: int64

Duplicates: 0


In [2]:
print(df[["Inventory Level","Units Sold","Units Ordered","Demand"]].describe())

       Inventory Level    Units Sold  Units Ordered        Demand
count     76000.000000  76000.000000   76000.000000  76000.000000
mean        301.062842     88.827316      89.090645    104.317158
std         226.510161     43.994525     162.404627     46.964801
min           0.000000      0.000000       0.000000      4.000000
25%         136.000000     58.000000       0.000000     71.000000
50%         227.000000     84.000000       0.000000    100.000000
75%         408.000000    114.000000     121.000000    133.000000
max        2267.000000    426.000000    1616.000000    430.000000


In [3]:
print(df[["Inventory Level","Units Sold","Units Ordered","Demand"]].corr())

                 Inventory Level  Units Sold  Units Ordered    Demand
Inventory Level         1.000000    0.234590      -0.034698  0.126618
Units Sold              0.234590    1.000000       0.433943  0.833421
Units Ordered          -0.034698    0.433943       1.000000  0.511963
Demand                  0.126618    0.833421       0.511963  1.000000


In [4]:
df["Date"] = pd.to_datetime(df["Date"])

print("Date range:")
print(df["Date"].min(), "to", df["Date"].max())

Date range:
2022-01-01 00:00:00 to 2024-01-30 00:00:00


In [5]:
print("Inventory < Demand:", (df["Inventory Level"] < df["Demand"]).sum())

print("Stockout rate:",(df["Inventory Level"] < df["Demand"]).mean())

Inventory < Demand: 10394
Stockout rate: 0.13676315789473684


In [6]:
print("Unique stores:", df["Store ID"].nunique())
print("Unique products:", df["Product ID"].nunique())
print("Store-product combinations:",df.groupby(["Store ID", "Product ID"]).ngroups)

Unique stores: 5
Unique products: 20
Store-product combinations: 100


In [7]:
df = df.sort_values(["Store ID", "Product ID", "Date"]).reset_index(drop=True)

print(df.groupby(["Store ID", "Product ID"])["Date"].diff().value_counts().head())

Date
1 days    75900
Name: count, dtype: int64


In [8]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

In [9]:
from src.data_preprocessing import preprocess_data
from src.feature import create_features

df = preprocess_data(df)
df_features = create_features(df)

print("Shape:", df_features.shape)
print("\nEngineered features:")
print(["Previous_Day_Sales","Avg_7_Day_Sales","Previous_Day_Demand"])

Shape: (75200, 20)

Engineered features:
['Previous_Day_Sales', 'Avg_7_Day_Sales', 'Previous_Day_Demand']


In [10]:
print("Target distribution:")
print(df_features["Stockout_Next_Day"].value_counts())

print("\nTarget proportion:")
print(df_features["Stockout_Next_Day"].value_counts(normalize=True))

Target distribution:
Stockout_Next_Day
0    65012
1    10188
Name: count, dtype: int64

Target proportion:
Stockout_Next_Day
0    0.864521
1    0.135479
Name: proportion, dtype: float64
